# Post-Deployment Notebook

Runs post-deployment scripts in a controlled, idempotent way.  
Each script is identified by a `(release_version, script_id)` pair.  

**Execution Logic:**  
Before running a script, the notebook checks the **DeploymentScriptLog** Delta table for the highest completed `script_id` for that `release_version`.  
The script is executed **only if** its `script_id` is higher than the max completed value in the lakehouse.  
Scripts with `script_id` ≤ max are skipped automatically.

In [ ]:
# ---------------------------------------------------------------------------
# Parameters  (injected by the pipeline or set manually for local runs)
# ---------------------------------------------------------------------------
environment = "DEV"   # overridden at runtime by notebook_executor parameters

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import col
from delta.tables import DeltaTable
from datetime import datetime, timezone
from notebookutils import mssparkutils

CONTROL_TABLE = "DeploymentScriptLog"

## Control Table

`DeploymentScriptLog` tracks every script that has been executed.  
Schema: `release_version`, `script_id`, `description`, `status` (`Completed` / `Failed`), `executed_at`, `error_message`.

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
        release_version  STRING   NOT NULL,
        script_id        INT      NOT NULL,
        description      STRING,
        status           STRING   NOT NULL,
        executed_at      TIMESTAMP,
        error_message    STRING
    )
    USING DELTA
""")

print(f"Control table ready: {CONTROL_TABLE}")

## Helper Functions

In [ ]:
def get_max_completed_script_id(release_version: str) -> int:
    """Return the highest completed script_id for a given release_version, or -1 if none."""
    result = spark.sql(f"""
        SELECT MAX(script_id) AS max_id
        FROM {CONTROL_TABLE}
        WHERE release_version = '{release_version}'
          AND status = 'Completed'
    """).collect()
    
    max_id = result[0]["max_id"]
    return max_id if max_id is not None else -1


def should_run_script(release_version: str, script_id: int) -> bool:
    """Return True if script_id is higher than the max completed script_id in lakehouse."""
    max_completed = get_max_completed_script_id(release_version)
    return script_id > max_completed


def log_script(release_version: str, script_id: int, description: str,
               status: str, error_message: str = None):
    """Insert or update a log row for the given script."""
    now = datetime.now(timezone.utc)
    error_val = f"'{error_message}'" if error_message else "NULL"

    spark.sql(f"""
        MERGE INTO {CONTROL_TABLE} AS target
        USING (
            SELECT
                '{release_version}'  AS release_version,
                {script_id}          AS script_id,
                '{description}'      AS description,
                '{status}'           AS status,
                TIMESTAMP '{now.strftime('%Y-%m-%d %H:%M:%S')}' AS executed_at,
                {error_val}          AS error_message
        ) AS source
        ON  target.release_version = source.release_version
        AND target.script_id       = source.script_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


def run_script(release_version: str, script_id: int, description: str, fn):
    """
    Guard-execute a script function.
    Skips execution when script_id is not higher than the max completed script_id in lakehouse.
    Logs the result (Completed / Failed) after each run.
    """
    label = f"[{release_version} / script {script_id}] {description}"
    
    max_completed = get_max_completed_script_id(release_version)

    if not should_run_script(release_version, script_id):
        print(f"  SKIP  {label}  (max completed script_id: {max_completed})")
        return

    print(f"  RUN   {label}  (max completed script_id: {max_completed})")
    try:
        fn()
        log_script(release_version, script_id, description, "Completed")
        print(f"  OK    {label}")
    except Exception as exc:
        error_msg = str(exc).replace("'", "''")   # escape single quotes for SQL
        log_script(release_version, script_id, description, "Failed", error_msg)
        print(f"  FAIL  {label}  →  {exc}")
        raise  # re-raise so the notebook fails visibly in the pipeline

## Script Registry

Add every post-deployment script here as a dict with keys:
- `release_version` – semantic version string, e.g. `"1.0"`
- `script_id` – integer, unique within a release
- `description` – human-readable label shown in logs
- `fn` – a zero-argument callable containing the script logic

Scripts are executed in list order. Scripts with `script_id` ≤ the max completed `script_id` in the lakehouse are automatically skipped.

In [ ]:
# ---------------------------------------------------------------------------
# Script implementations
# Each function contains the actual work for one post-deployment step.
# Keep functions self-contained; raise an exception to signal failure.
# ---------------------------------------------------------------------------



def script_1_0_1():
    """Table DDL scripts"""
    # Execute another notebook with optional parameters
    notebook_name = "DLL_Notebook_1_0_1"  # Change to your notebook name
    timeout_seconds = 600
    
    # Option 1: Pass parameters
    # parameters = {
    #     "environment": environment  # Pass current environment to the notebook
    # }
    
    parameters = {}
    
    print(f"  → Executing notebook: {notebook_name}")
    result = mssparkutils.notebook.run(
        notebook_name, 
        timeout_seconds, 
        parameters
    )
    print(f"  ✓ Notebook completed with result: {result}")


# ---------------------------------------------------------------------------
# REGISTRY  ← add your scripts here
# ---------------------------------------------------------------------------
SCRIPTS = [
    {
        "release_version": "1.0",
        "script_id": 1,
        "description": "Seed AllowedItemKind lookup table",
        "fn": script_1_0_1,
    },
    # ── add new scripts below this line ──────────────────────────────────
]

## Execute Scripts

In [ ]:
print(f"Post-deployment run  |  environment={environment}  |  scripts={len(SCRIPTS)}")
print("-" * 70)

failed = []
for script in SCRIPTS:
    try:
        run_script(
            release_version=script["release_version"],
            script_id=script["script_id"],
            description=script["description"],
            fn=script["fn"],
        )
    except Exception:
        failed.append(script)

print("-" * 70)
if failed:
    labels = ", ".join(f"{s['release_version']}/{s['script_id']}" for s in failed)
    raise RuntimeError(f"Post-deployment finished with {len(failed)} failure(s): {labels}")

print("All scripts completed successfully.")

## Audit: View Control Table

In [ ]:
spark.sql(f"""
    SELECT release_version, script_id, description, status, executed_at, error_message
    FROM   {CONTROL_TABLE}
    ORDER  BY release_version, script_id
""").show(truncate=False)